# 04 — Feature Engineering
Raw columns වලින් analysis සඳහා ප්‍රයෝජනවත් transaction සහ customer features නිර්මාණය කරමු.

## ඉගෙනුම් අරමුණු
- LineValue සහ calendar features නිර්මාණය කිරීම
- `groupby().agg()` මගින් customer summaries සෑදීම
- `np.where()` conditional segmentation තේරුම් ගැනීම

In [ ]:
import numpy as np
from retail_analysis.data_loader import load_retail_data
from retail_analysis.cleaning import clean_transactions, split_transaction_sets
from retail_analysis.features import add_transaction_features, create_customer_features

cleaned, _ = clean_transactions(load_retail_data())
sales, _, _ = split_transaction_sets(cleaned)
sales = add_transaction_features(sales)
sales[['Quantity', 'UnitPrice', 'LineValue', 'YearMonth', 'DayOfWeek', 'Hour']].head()

Pandas columns අතර multiplication vectorized නිසා row-by-row loop අවශ්‍ය නැහැ. `YearMonth` එකේ year ද ඇති නිසා වෙනස් වසරවල එකම month එක එකට මිශ්‍ර නොවේ.

In [ ]:
customers = create_customer_features(sales)
customers.head(10)

`groupby('CustomerID').agg(...)` customer එකකට rows එකතු කරයි. `nunique()` එක invoice lines නොව unique orders ගණන් කරයි. Recency reference එක latest eligible sales date එකෙන් දින එකක් පසු දිනයයි. Features refunds අඩු කළ net values නොවේ.

In [ ]:
# np.where: condition True/False අනුව label එක vectorized ලෙස තෝරයි
np.where(customers['OrderCount'].eq(1), 'One-time buyer', 'Repeat buyer')[:10]

In [ ]:
customers.groupby('CustomerSegment').agg(
    CustomerCount=('CustomerID', 'count'),
    AveragePurchaseValue=('TotalPurchaseValue', 'mean')
)

## Interpretation සහ පුහුණු ප්‍රශ්න
Anonymous transactions sales totals වල තිබුණත් CustomerID නැති නිසා customer table එකේ නැහැ. AverageOrderValue = TotalPurchaseValue / unique OrderCount.

1. One-time සහ repeat customer counts compare කරන්න.
2. එක invoice එකේ lines කිහිපයක් ඇති customer කෙනෙකුගේ OrderCount verify කරන්න.
3. RecencyDays අඩුම customers කවුද?